In [1]:
import warnings
import os
import sys
import yaml

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [2]:
sys.path.append(os.path.abspath('..'))
from scripts.evaluate import evaluate_classifier

CONFIG_PATH = '../configs/paths.yaml'

with open(CONFIG_PATH, 'r') as file:
    paths = yaml.safe_load(file)

df_train = pd.read_csv(paths['features_data']['train_data'])
df_test  = pd.read_csv(paths['features_data']['test_data'])

print(f"Train shape: {df_train.shape}")
print(f"Test shape : {df_test.shape}")
df_train.head()

Train shape: (79219, 38)
Test shape : (20124, 38)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,glipizide-metformin,change,diabetesMed,diag_1_group,diag_2_group,diag_3_group,medical_specialty_group,total_prior_visits,high_utilizer,A1C_clinical_response,readmitted_binary
0,Caucasian,Female,0,Other_Type,Other_Discharge,Referral,1,Unknown,41,0,1,0,0,0,1,No,No,No,No,No,No,No,No,No,No,0,No,No,No,No,Diabetes,Unknown,Unknown,Pediatrics,0,0,Not_Measured,0
1,Caucasian,Female,1,Emergency,Discharged_Home,Emergency_Room,3,Unknown,59,0,18,0,0,0,9,No,No,No,No,No,No,No,No,No,No,3,No,No,Yes,Yes,Other,Diabetes,Other,Missing,0,0,Not_Measured,0
2,AfricanAmerican,Female,2,Emergency,Discharged_Home,Emergency_Room,2,Unknown,11,5,13,2,0,1,6,No,No,No,No,No,Yes,No,No,No,No,0,No,No,No,Yes,Other,Diabetes,Other,Missing,3,1,Not_Measured,0
3,Caucasian,Male,3,Emergency,Discharged_Home,Emergency_Room,2,Unknown,44,1,16,0,0,0,7,No,No,No,No,No,No,No,No,No,No,3,No,No,Yes,Yes,Other,Diabetes,Circulatory,Missing,0,0,Not_Measured,0
4,Caucasian,Male,5,Urgent,Discharged_Home,Referral,3,Unknown,31,6,16,0,0,0,9,No,No,No,No,No,No,No,No,No,No,1,No,No,No,Yes,Circulatory,Circulatory,Diabetes,Missing,0,0,Not_Measured,0


In [3]:
X_train = df_train.drop(columns=['readmitted_binary'])
X_test  = df_test.drop(columns=['readmitted_binary'])

y_train = df_train['readmitted_binary']
y_test  = df_test['readmitted_binary']

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")

X_train: (79219, 37)  |  y_train: (79219,)
X_test : (20124, 37)   |  y_test : (20124,)


In [4]:
from scripts.preprocessor import get_preprocessor

preprocessor = get_preprocessor()

In [5]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('gradientboost',GradientBoostingClassifier(
        random_state=32
    ))
])

default_result = evaluate_classifier(
    'Gradient Boost (default)',
    pipeline,
    X_train, X_test,
    y_train, y_test
)

print("GradientBoost — Default Parameters")
print(classification_report(y_test, default_result['_test_pred']))

GradientBoost — Default Parameters
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     17783
           1       0.54      0.01      0.02      2341

    accuracy                           0.88     20124
   macro avg       0.71      0.50      0.48     20124
weighted avg       0.84      0.88      0.83     20124



In [ ]:
param_grid = {
    "gradientboost__n_estimators": [100, 200],
    "gradientboost__learning_rate": [0.01, 0.1, 0.2],
    "gradientboost__max_depth": [2, 3, 5],
    "gradientboost__subsample": [0.8, 1.0]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    verbose=3
)

print("Running coarse grid search...")
grid.fit(X_train, y_train)



Running coarse grid search...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV 1/5] END gradientboost__learning_rate=0.01, gradientboost__max_depth=2, gradientboost__n_estimators=100, gradientboost__subsample=0.8;, score=0.000 total time= 1.5min
[CV 2/5] END gradientboost__learning_rate=0.01, gradientboost__max_depth=2, gradientboost__n_estimators=100, gradientboost__subsample=0.8;, score=0.000 total time= 1.3min
[CV 3/5] END gradientboost__learning_rate=0.01, gradientboost__max_depth=2, gradientboost__n_estimators=100, gradientboost__subsample=0.8;, score=0.000 total time= 1.6min
[CV 4/5] END gradientboost__learning_rate=0.01, gradientboost__max_depth=2, gradientboost__n_estimators=100, gradientboost__subsample=0.8;, score=0.000 total time= 1.6min
[CV 5/5] END gradientboost__learning_rate=0.01, gradientboost__max_depth=2, gradientboost__n_estimators=100, gradientboost__subsample=0.8;, score=0.000 total time= 1.4min
[CV 1/5] END gradientboost__learning_rate=0.01, gradi

In [ ]:
print("\nBest Parameters (coarse):", grid_coarse.best_params_)
print("Best CV F1 Score (coarse):", round(grid_coarse.best_score_, 4))